# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Type: Scoring / Ranking** (with a classification model underneath)

My lane is Refresh / Content Opportunity Scoring. The end deliverable
is a ranked queue — pages ordered by "how worth reviewing" they are —
not a simple yes/no label. To build that ranking, I'll train a binary
classifier (predicting whether a page is declining) and use its output
probability as the ranking score, the same approach the Week 1 starter
notebooks used to rank pages by model probability and measure
Precision@K.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target/proxy:** `is_declining_label`, defined as `trend_direction == "down"`.

This label comes from a **defined rule on the current window**, not an
observed future outcome — it's a proxy, not the real thing I ultimately
care about. A stronger, less proxy-dependent version (per the lane guide)
would use a future-window label: features from the prior 90 days
predicting decline over the next 30 days. I'm starting with the proxy
because it's what the starter data supports, and I can move toward a
future-window label once I pull from the full warehouse release.
Section 3 — Success metric:

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50.**

Of the top 50 pages my ranking flags first, how many are actually
declining? This matches the real decision-maker's constraint: a content
team can only act on a limited number of pages per sprint, so what
matters is whether the top of the list is trustworthy — not overall
accuracy across all 30,000 pages, most of which nobody will ever review.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/imnotparama/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Unit of analysis: one row = one content page (content_id)")
print(f"Total rows: {df.shape[0]}")
df[["content_id", "impressions_90d", "days_since_last_update",
    "trend_direction", "avg_position", "word_count"]].head(5)
# Sketch of the target column
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(df["is_declining_label"].value_counts())
print(f"\n% declining: {df['is_declining_label'].mean():.1%}")
df[["content_id", "trend_direction", "is_declining_label"]].head(10)

Working dir: /content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship
Unit of analysis: one row = one content page (content_id)
Total rows: 30000
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

% declining: 54.2%


,content_id,trend_direction,is_declining_label
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
6,content_9a34b442b552,down,1
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,1
9,content_c27558df2b0c,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why ML beats a fixed rule:** In Week 1, a hand-written rule (stale
AND visible) scored Precision@50 of 0.240, while a random forest scored
0.740 — roughly 3x better, computed live on the same data. A fixed rule
can only combine a couple of signals with thresholds I pick by hand
(e.g. "180+ days stale AND 500+ impressions"). Real decline is messier
than that: it depends on interactions between freshness, position, CTR,
word count, and traffic level all at once, in ways that shift
non-linearly. A model can learn those interactions directly from data
instead of me guessing which combination of if-statements captures it.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.